In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

# 02 — The ProcessParameters Object

The `ProcessParameters` object defines the machine configuration. It is inherently vectorized — any field can be a scalar or a NumPy array, enabling single-point simulations and full N-dimensional parameter sweeps from the same API.

## Section 1 — Single-point (scalar) parameters

In [2]:
from lpbf_maps.parameters import ProcessParameters

params = ProcessParameters(
    laser_power=200.0,
    scan_speed=1.0,
    beam_radius=50e-6,
    hatch_spacing=80e-6,
    layer_thickness=30e-6,
    ambient_temperature=298.0
)

print(f"Shape: {params.shape}")
print(f"Is Vectorized: {params.is_vectorized}")

for field in ["laser_power", "scan_speed", "beam_radius"]:
    val = getattr(params, field)
    print(f"{field}: {val} — shape: {val.shape}")

Shape: ()
Is Vectorized: False
laser_power: 200.0 — shape: ()
scan_speed: 1.0 — shape: ()
beam_radius: 5e-05 — shape: ()


## Section 2 — 1D vectorized grid (scan speed sweep)

In [3]:
import numpy as np

scan_speeds = np.linspace(0.5, 2.0, 10)
params_1d = ProcessParameters(
    laser_power=200.0,
    scan_speed=scan_speeds,
    beam_radius=50e-6
)

print(f"Shape: {params_1d.shape}")
print(f"Is Vectorized: {params_1d.is_vectorized}")

Shape: (10,)
Is Vectorized: True


## Section 3 — 2D evaluation grid (P vs v)

In [4]:
laser_powers = np.linspace(100, 300, 5)[:, np.newaxis]
scan_speeds_2d = np.linspace(0.5, 2.0, 6)[np.newaxis, :]

params_2d = ProcessParameters(
    laser_power=laser_powers,
    scan_speed=scan_speeds_2d,
    beam_radius=50e-6
)

print(f"Shape: {params_2d.shape} — the classic 2D processing map grid")

Shape: (5, 6) — the classic 2D processing map grid


## Section 4 — The wavelength field

The `wavelength` field (in metres) enables the `Material.calculate_absorptivity()` to compute material-specific laser coupling before or during a simulation.

In [5]:
params_wv = ProcessParameters(
    laser_power=200.0,
    scan_speed=1.0,
    beam_radius=50e-6,
    wavelength=1.064e-6
)
print(f"Wavelength: {params_wv.wavelength} m")

Wavelength: 1.064e-06 m


## Section 5 — The `.get_point()` method

In [6]:
point_params = params_2d.get_point((2, 3))
print(f"Point shape: {point_params.shape}")
print(f"Point Laser Power: {point_params.laser_power}")
print(f"Point Scan Speed: {point_params.scan_speed}")

Point shape: ()
Point Laser Power: 200.0
Point Scan Speed: 1.4


## Section 6 — Validation guards

In [7]:
try:
    bad_params = ProcessParameters(laser_power=-100, scan_speed=1.0, beam_radius=50e-6)
except ValueError as e:
    print(f"Caught ValueError: {e}")

Caught ValueError: Laser power must be strictly positive.
